In [ ]:
# Part (a)
import math
from IPython.display import Image, display
from jpeg_utils import (
    create_canvas,
    draw_line,
    fill_rect,
    draw_point,
    draw_text,
    save_grayscale_jpeg,
)

def draw_dashed_horizontal(canvas, x0, x1, y, color, dash=8, gap=6):
    x = x0
    while x < x1:
        x_end = min(x + dash, x1)
        draw_line(canvas, x, y, x_end, y, color)
        x += dash + gap


def save_acf_jpg(acf_values, n_eff, filename, max_lag):
    width, height = 800, 400
    margin_left, margin_right, margin_top, margin_bottom = 70, 20, 40, 70
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    canvas = create_canvas(width, height)

    draw_line(canvas, margin_left, margin_top + plot_height, margin_left + plot_width, margin_top + plot_height, 0)
    draw_line(canvas, margin_left, margin_top, margin_left, margin_top + plot_height, 0)

    conf = 1.96 / math.sqrt(n_eff)

    def value_to_y(value):
        return margin_top + (1 - (value + 1) / 2) * plot_height

    zero_y = value_to_y(0.0)
    draw_line(canvas, margin_left, zero_y, margin_left + plot_width, zero_y, 180)
    draw_dashed_horizontal(canvas, margin_left, margin_left + plot_width, value_to_y(conf), 120)
    draw_dashed_horizontal(canvas, margin_left, margin_left + plot_width, value_to_y(-conf), 120)

    bar_half_width = (plot_width / max_lag) * 0.35
    for lag in range(1, max_lag + 1):
        value = acf_values[lag]
        x_center = margin_left + (lag - 0.5) * (plot_width / max_lag)
        y_value = value_to_y(value)
        draw_line(canvas, x_center, zero_y, x_center, y_value, 80)
        fill_rect(canvas, x_center - bar_half_width, zero_y, x_center + bar_half_width, y_value, 80)
        label = str(lag)
        label_x = int(x_center) - (len(label) * 6) // 2
        draw_text(canvas, label_x, int(margin_top + plot_height + 12), label, 0)

    draw_text(canvas, margin_left + plot_width // 2 - 90, 10, 'ACF OF AR(3) RESIDUALS', 0)
    draw_text(canvas, margin_left + plot_width // 2 - 10, height - 50, 'LAG', 0)
    draw_text(canvas, 10, margin_top - 20, 'AUTOCORRELATION', 0)

    save_grayscale_jpeg(filename, canvas)


def save_qq_plot_jpg(points, slope, intercept, filename):
    width, height = 800, 400
    margin_left, margin_right, margin_top, margin_bottom = 70, 30, 40, 70
    plot_width = width - margin_left - margin_right
    plot_height = height - margin_top - margin_bottom
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)
    x_pad = (x_max - x_min) * 0.1 or 0.1
    y_pad = (y_max - y_min) * 0.1 or 0.1
    x_min -= x_pad
    x_max += x_pad
    y_min -= y_pad
    y_max += y_pad

    canvas = create_canvas(width, height)

    def to_pixel(x, y):
        px = margin_left + (x - x_min) / (x_max - x_min) * plot_width
        py = margin_top + (y_max - y) / (y_max - y_min) * plot_height
        return px, py

    draw_line(canvas, margin_left, margin_top + plot_height, margin_left + plot_width, margin_top + plot_height, 0)
    draw_line(canvas, margin_left, margin_top, margin_left, margin_top + plot_height, 0)

    line_x0, line_y0 = x_min, slope * x_min + intercept
    line_x1, line_y1 = x_max, slope * x_max + intercept
    px0, py0 = to_pixel(line_x0, line_y0)
    px1, py1 = to_pixel(line_x1, line_y1)
    draw_line(canvas, px0, py0, px1, py1, 80)

    for x, y in points:
        px, py = to_pixel(x, y)
        draw_point(canvas, px, py, 2, 60)

    draw_text(canvas, margin_left + plot_width // 2 - 85, 10, 'QQ PLOT OF AR(3) RESIDUALS', 0)
    draw_text(canvas, margin_left + plot_width // 2 - 100, height - 50, 'THEORETICAL QUANTILES', 0)
    draw_text(canvas, margin_left + 10, margin_top - 20, 'SAMPLE QUANTILES', 0)

    save_grayscale_jpeg(filename, canvas)


def load_series(path):
    values = []
    with open(path) as f:
        for token in f.read().split():
            try:
                values.append(float(token))
            except ValueError:
                continue
    return values


def build_regression_matrices(series, order):
    X = []
    y = []
    for t in range(order, len(series)):
        row = [1.0]
        for j in range(1, order + 1):
            row.append(series[t - j])
        X.append(row)
        y.append(series[t])
    return X, y


def solve_linear_system(A, b):
    n = len(A)
    aug = [A[i][:] + [b[i]] for i in range(n)]
    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(aug[r][col]))
        aug[col], aug[pivot_row] = aug[pivot_row], aug[col]
        pivot = aug[col][col]
        if abs(pivot) < 1e-12:
            raise ValueError('Singular matrix encountered while solving normal equations.')
        for j in range(col, n + 1):
            aug[col][j] /= pivot
        for r in range(n):
            if r == col:
                continue
            factor = aug[r][col]
            if factor == 0:
                continue
            for j in range(col, n + 1):
                aug[r][j] -= factor * aug[col][j]
    return [aug[i][n] for i in range(n)]


def fit_ar(series, order):
    X, y = build_regression_matrices(series, order)
    p = len(X[0])
    XtX = [[0.0] * p for _ in range(p)]
    XtY = [0.0] * p
    for row, target in zip(X, y):
        for j in range(p):
            XtY[j] += row[j] * target
            for k in range(p):
                XtX[j][k] += row[j] * row[k]
    beta = solve_linear_system(XtX, XtY)
    return beta, X, y


def compute_residuals(series, beta, order):
    residuals = [None] * order
    for t in range(order, len(series)):
        pred = beta[0]
        for j in range(1, order + 1):
            pred += beta[j] * series[t - j]
        residuals.append(series[t] - pred)
    return residuals


def autocorrelation(data, lag):
    m = len(data)
    mean = sum(data) / m
    denom = sum((x - mean) ** 2 for x in data)
    num = sum((data[i] - mean) * (data[i - lag] - mean) for i in range(lag, m))
    return num / denom


hare_counts = load_series('hare.dat')
hare_series = [math.sqrt(value) for value in hare_counts]
order = 3
beta, X_matrix, y_vector = fit_ar(hare_series, order)
residuals = compute_residuals(hare_series, beta, order)
filtered_residuals = [r for r in residuals if r is not None]
max_lag = 20
acf_values = [1.0] + [autocorrelation(filtered_residuals, lag) for lag in range(1, max_lag + 1)]
save_acf_jpg(acf_values, len(filtered_residuals), 'hare_residuals_acf.jpg', max_lag)
display(Image(filename='hare_residuals_acf.jpg'))


In [ ]:
# Part (b)import mathK = 9N = len(filtered_residuals)ljung_box_Q = N * (N + 2) * sum((acf_values[k] ** 2) / (N - k) for k in range(1, K + 1))print(f'Ljung-Box Q-statistic (K=9): {ljung_box_Q:.4f}')print(f'Degrees of freedom: {K - 3}')

In [ ]:
# Part (c)from statistics import NormalDistnonzero_residuals = [r for r in filtered_residuals if r != 0]signs = [1 if r > 0 else -1 for r in nonzero_residuals]runs = 1for i in range(1, len(signs)):    if signs[i] != signs[i - 1]:        runs += 1n_pos = sum(1 for s in signs if s == 1)n_neg = sum(1 for s in signs if s == -1)expected_runs = 1 + 2 * n_pos * n_neg / (n_pos + n_neg)variance_runs = (2 * n_pos * n_neg * (2 * n_pos * n_neg - n_pos - n_neg)) / (((n_pos + n_neg) ** 2) * (n_pos + n_neg - 1))z_runs = (runs - expected_runs) / math.sqrt(variance_runs)p_runs = 2 * (1 - NormalDist().cdf(abs(z_runs)))print(f'Runs count: {runs}')print(f'Expected runs: {expected_runs:.4f}')print(f'Z-statistic: {z_runs:.4f}')print(f'Two-sided p-value: {p_runs:.4f}')

In [ ]:
# Part (d)
from statistics import NormalDist
from IPython.display import Image, display

ndist = NormalDist()
sorted_residuals = sorted(filtered_residuals)
N = len(sorted_residuals)
qq_points = []
for i, value in enumerate(sorted_residuals, start=1):
    prob = (i - 0.375) / (N + 0.25)
    theor = ndist.inv_cdf(prob)
    qq_points.append((theor, value))

sum_x = sum(pt[0] for pt in qq_points)
sum_y = sum(pt[1] for pt in qq_points)
sum_xx = sum(pt[0] ** 2 for pt in qq_points)
sum_xy = sum(pt[0] * pt[1] for pt in qq_points)
slope = (N * sum_xy - sum_x * sum_y) / (N * sum_xx - sum_x ** 2)
intercept = (sum_y - slope * sum_x) / N

save_qq_plot_jpg(qq_points, slope, intercept, 'hare_residuals_qq.jpg')
display(Image(filename='hare_residuals_qq.jpg'))


In [ ]:
# Part (e)import mathimport randomfrom statistics import NormalDistndist = NormalDist()N = len(filtered_residuals)sorted_residuals = sorted(filtered_residuals)coefficients = [ndist.inv_cdf((i - 0.375) / (N + 0.25)) for i in range(1, N + 1)]norm_factor = math.sqrt(sum(c * c for c in coefficients))a_weights = [c / norm_factor for c in coefficients]mean_residual = sum(filtered_residuals) / Ndenom = sum((r - mean_residual) ** 2 for r in filtered_residuals)numer = sum(a * x for a, x in zip(a_weights, sorted_residuals)) ** 2W_statistic = numer / denomrandom.seed(2024)iterations = 5000count = 0for _ in range(iterations):    sample = sorted(random.gauss(0, 1) for _ in range(N))    mean_sample = sum(sample) / N    denom_sample = sum((s - mean_sample) ** 2 for s in sample)    numer_sample = sum(a * s for a, s in zip(a_weights, sample)) ** 2    if denom_sample == 0:        continue    W_sample = numer_sample / denom_sample    if W_sample <= W_statistic:        count += 1p_value = count / iterationsprint(f'Shapiro-Wilk W statistic (Monte Carlo approximation): {W_statistic:.4f}')print(f'Approximate p-value: {p_value:.4f}')